# Attention U-Net → BUSI — project notebook

Orchestration for Google Colab: **bootstrap → data → experiments → figures**.
All logic lives in the `busi/` package; this notebook only orchestrates.
The control panel is `busi/config.py` — switch model/loss/params there or per run below.

See `../PLAN.md` for the milestone experiment matrix (Core → CBAM → scSE).

## 1. Bootstrap — Colab only

On a **fresh Colab runtime** (a blank cloud machine), run this cell first: it
clones the repo and installs the dependencies onto that machine.

**Running on a local kernel** (your own `.venv`, e.g. in VS Code)? **Skip this
cell** — just make sure the notebook's working directory is the repo root (the
folder containing `busi/`).

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yuval-Naim/DL_attention_busi.git"   # public — no token needed
REPO_DIR = "DL_attention_busi"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt", "-i", "https://pypi.org/simple"], check=True)
    # (Optional, for long runs) mount Drive to persist checkpoints across disconnects:
    #   from google.colab import drive; drive.mount("/content/drive")

print("cwd:", os.getcwd())

## 2. Sanity check

In [ ]:
import torch, busi
from busi.config import Config
from busi import train as T
print("busi", busi.__version__, "| torch", torch.__version__, "| device", T.get_device())

## 3. Data — verify (or download)

BUSI is **not** committed/submitted. This cell first checks `data/BUSI/`:

- **If it's already there** (you placed the `benign/ malignant/ normal/` folders, or
  a previous run downloaded them) → it just **verifies and moves on — no Kaggle needed.**
- **If it's missing** → it downloads from Kaggle.

To enable the download: get a Kaggle token (kaggle.com → *Settings → API → Create
New Token* → `kaggle.json`) and provide it in **one** of these — **no key is written
into this notebook**:
- **VS Code / local:** put `kaggle.json` at `~/.kaggle/kaggle.json`, or set
  `KAGGLE_USERNAME` / `KAGGLE_KEY`.
- **Colab (browser):** add its contents as a Secret named `KAGGLE_JSON`.

(If your data lives elsewhere, set `DATA_ROOT` in the next cell to that path.)
Expected: **780** images (437/210/133); split saved to `splits/split.json`.

*Dataset: Al-Dhabyani et al., "Dataset of breast ultrasound images", Data in Brief
28 (2020), CC BY 4.0.*

In [ ]:
import os, pathlib, subprocess, sys
from busi import experiment as E
from busi.data import list_busi_samples

DATA_ROOT = "data/BUSI"     # where the dataset will be downloaded to

def _ensure_kaggle_creds():
    """Find Kaggle credentials WITHOUT hardcoding them in the notebook."""
    kp = pathlib.Path(os.path.expanduser("~/.kaggle/kaggle.json"))
    if kp.exists() or (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        return                                   # VS Code / local path
    try:                                         # Colab (browser) secret
        from google.colab import userdata
        kp.parent.mkdir(exist_ok=True)
        kp.write_text(userdata.get("KAGGLE_JSON")); os.chmod(kp, 0o600)
    except Exception:
        raise RuntimeError(
            "No Kaggle credentials found. See the cell above: add a KAGGLE_JSON "
            "Colab secret, put kaggle.json in ~/.kaggle/, or set "
            "KAGGLE_USERNAME/KAGGLE_KEY.")

if not os.path.isdir(DATA_ROOT):
    _ensure_kaggle_creds()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "datasets", "download", "-d",
                    "aryashah2k/breast-ultrasound-images-dataset", "-p", "data", "--unzip"], check=True)
    if os.path.isdir("data/Dataset_BUSI_with_GT"):   # Kaggle unzips to this name
        os.rename("data/Dataset_BUSI_with_GT", DATA_ROOT)

cfg = Config(data_root=DATA_ROOT)
samples = list_busi_samples(cfg.data_root, classes=tuple(cfg.classes))
print("total samples:", len(samples))               # expect 780
split = E.get_or_make_split(cfg)
print({k: len(v) for k, v in split.items()})

## 4. Core experiments (baseline vs attention)

Trains the plain U-Net (C1) and the Attention U-Net (C2), each over 3 seeds, and
aggregates mean±std. This is the heavy step — **run on a Colab GPU**. Set `QUICK=True`
for a fast local smoke (1 seed, few epochs).

In [ ]:
QUICK = False    # True -> 1 seed + few epochs, for a fast local smoke
base = Config(data_root=DATA_ROOT, epochs=(3 if QUICK else 100),
              seeds=([42] if QUICK else [42, 1, 7]))

# Core = unet, attention_unet. Desired adds cbam_unet. Stretch adds scse_unet.
MODELS = ["unet", "attention_unet", "cbam_unet", "scse_unet"]
results = [E.run_seeds(name, cfg=base, seeds=base.seeds) for name in MODELS]

## 5. Results table & qualitative figures

In [ ]:
print(E.make_results_table(results))

# Attention-map figures for the attention model (best-seed checkpoint).
att = next(r for r in results if r["model"] == "attention_unet")
cfg_att = E._cfg_for(base, "attention_unet", att["seeds"][0])
ckpt = f"{cfg_att.checkpoints_dir}/{cfg_att.experiment_name}.pt"
figs = E.save_prediction_figures(cfg_att, ckpt, "results/figures", n=6)
print("saved figures:", figs)

## 6. Milestones & the Focal-Tversky run

`MODELS` already covers **Core** (`unet`, `attention_unet`), **Desired** (`cbam_unet`)
and **Stretch** (`scse_unet`).

Optional stretch — a loss study on the best attention variant (small imbalanced
lesions):

```python
ft = E.run_seeds("attention_unet",
                 cfg=Config(loss_name="focal_tversky", epochs=base.epochs),
                 seeds=base.seeds)
```

Results JSON land in `results/`, figures in `results/figures/`, checkpoints in
`checkpoints/` (on Colab, keep these on Drive so a disconnect doesn't lose them).